**INSTALL DEPENDENCIES**

In [1]:
!pip install transformers accelerate pillow tqdm

**IMPORTS**

In [2]:
import os
import json
from PIL import Image
from tqdm import tqdm
import torch

from transformers import Blip2Processor, Blip2ForConditionalGeneration

**LOAD MODEL**

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP-2 model...")

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
).to(device)

print("Model loaded on", device)

Loading BLIP-2 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded on cuda


**DRIVE MOUNT**

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**SET PATHS**

In [10]:
IMAGE_FOLDER = "/content/drive/MyDrive/BanglaVision/bd_images"
OUTPUT_JSON = "/content/drive/MyDrive/BanglaVision/bd_captions_en.json"

**CAPTION FUNCTION**

In [11]:
def generate_caption(image_path):
    try:
        image = Image.open(image_path).convert("RGB")

        inputs = processor(images=image, return_tensors="pt").to(device, torch.float16)

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30
        )

        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return caption

    except Exception as e:
        return "__ERROR__"

**COLLECT ALL IMAGES**

In [12]:
image_paths = []

for root, dirs, files in os.walk(IMAGE_FOLDER):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            image_paths.append(os.path.join(root, file))

print("Total images found:", len(image_paths))

Total images found: 11593


**GENERATE CAPTIONS**

In [13]:
results = []

for img_path in tqdm(image_paths):

    caption = generate_caption(img_path)

    results.append({
        "image": img_path,
        "caption_en": caption
    })

100%|██████████| 11593/11593 [3:05:23<00:00,  1.04it/s]


**SAVE RESULTS**

In [14]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Captions saved at:", OUTPUT_JSON)

Captions saved at: /content/drive/MyDrive/BanglaVision/bd_captions_en.json
